# SenticDiT v5 — Kaggle runner

Companion to *Revisiting Emotional TTS Fine-Tuning on MELD: What Actually Matters*.
LoRA fine-tuning of `meituan-longcat/LongCat-AudioDiT-1B` on MELD, with the full v5 evaluation
(zero-shot control, corrected MCD, UTMOS, pitch-drift mechanism, Whisper WER, two SER models,
speaker consistency, Config D).

All code lives in the [SenticDiT repo](https://github.com/HellDragger/SenticDiT) (`senticdit/`).
This notebook only fetches it, sets the configuration, and runs it.

**Kaggle setup:** GPU T4 ×2 · Internet **on** · add inputs
`zaber666/meld-dataset`, `aryansharma26/meld-audio`, and (to resume) a previous run's output.

## 0 · Fetch the code & install dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/HellDragger/SenticDiT.git"
REPO_REF = "main"                          # branch or tag to run
REPO_DIR = "/kaggle/working/SenticDiT"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
sys.path.insert(0, REPO_DIR)

from senticdit.setup_env import prepare_environment
prepare_environment()   # pip installs + clones LongCat-AudioDiT (the `audiodit` package)

## 1 · Configuration

Every field and its rationale is documented in `senticdit/config.py`; anything not set here keeps
its default. To replicate a run, bump `run_tag` and `eval_seed` — results accumulate in `run_log.csv`.

In [ ]:
from senticdit import Config

CFG = Config(
    # --- data / output ----------------------------------------------------------------------
    kaggle_input_dir = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw",
    audio_dir        = "/kaggle/input/datasets/aryansharma26/meld-audio/MELD_audio",
    output_dir       = "/kaggle/working/audiodit_fixed",

    # --- main run: resume a previous session's checkpoints (None = train from scratch) ------
    resume_search_dir = "/kaggle/input/notebooks/aryansharma26/senticdit/audiodit_fixed/checkpoints/r16_attn_only",
    num_epochs        = 20,
    checkpoint_steps  = 200,

    # --- ablations: skip and reuse the selections already found ------------------------------
    skip_ablations              = True,
    best_lora_rank_override     = 16,     # cosmetic when resuming — the checkpoint carries the rank
    best_include_adaln_override = False,

    # --- generation --------------------------------------------------------------------------
    cfg_strength = 2.0,
    cfm_steps    = 16,

    # --- evaluation switches -----------------------------------------------------------------
    run_zero_shot_baseline   = True,
    run_ser_real_audio_check = True,
    run_utmos                = True,
    run_wer                  = True,
    run_emotion2vec          = True,
    run_speaker_consistency  = True,
    verify_weight_loading    = True,

    # --- Config D (data quantity at fixed compute, ~4.3 h) -----------------------------------
    run_config_d       = True,
    config_d_max_steps = 2000,

    # --- reproducibility ---------------------------------------------------------------------
    run_tag   = "v4_run1",
    eval_seed = 1234,
    seed      = 42,

    debug_subset = None,   # e.g. 200 for a fast smoke test of the whole pipeline
)

## 2 · Run

Runs every stage in order and writes all CSVs, WAVs and figures to `CFG.output_dir`.
To debug a single stage, call the methods one at a time instead of `run_all()`
(`prepare_data`, `load_model`, `baseline_sample`, `zero_shot`, `run_ablations`, `train_main`,
`cfg_sweep`, `final_eval`, `ser_eval`, `paraphrase`, `demos`, `mcd_eval`, `utmos_eval`,
`wer_eval`, `emotion2vec_eval`, `speaker_eval`, `config_d`, `write_report`).

In [ ]:
from senticdit.pipeline import SenticDiTRun

run = SenticDiTRun(CFG)
paper_df = run.run_all()
paper_df